# Dataset Check

mipnerf360 데이터셋의 이미지 크기, 카메라 파라미터, 씬 구성을 확인하는 노트북.

In [ ]:
# === 셀 1: 설정 ===
import os, glob, struct
import numpy as np
from PIL import Image
from collections import namedtuple

# ========== 여기만 수정 ==========
SCENE_NAME = "bicycle"
# 사용 가능한 씬: bicycle, bonsai, counter, garden, kitchen, room, stump
# ================================

DATASET_BASE = "/home/daeho/storage/3dgs_sba/datasets/mipnerf360"
SCENE_DIR = os.path.join(DATASET_BASE, SCENE_NAME)

print(f"SCENE: {SCENE_NAME}")
print(f"PATH : {SCENE_DIR}")
print(f"\n포함된 폴더/파일:")
for item in sorted(os.listdir(SCENE_DIR)):
    full = os.path.join(SCENE_DIR, item)
    if os.path.isdir(full):
        n = len(os.listdir(full))
        print(f"  {item}/ ({n} files)")
    else:
        size = os.path.getsize(full)
        print(f"  {item} ({size:,} bytes)")

In [ ]:
# === 셀 2: 이미지 크기 확인 ===

img_dir = os.path.join(SCENE_DIR, "images")
img_files = sorted(glob.glob(os.path.join(img_dir, "*")))

sizes = {}
file_sizes = []
for f in img_files:
    img = Image.open(f)
    w, h = img.size
    key = f"{w}x{h}"
    sizes[key] = sizes.get(key, 0) + 1
    file_sizes.append(os.path.getsize(f))

print(f"=== images/ ({len(img_files)}장) ===\n")
print(f"해상도 분포:")
for res, count in sorted(sizes.items(), key=lambda x: -x[1]):
    print(f"  {res}: {count}장")

print(f"\n파일 크기:")
print(f"  Min : {min(file_sizes)/1024/1024:.1f} MB")
print(f"  Max : {max(file_sizes)/1024/1024:.1f} MB")
print(f"  Mean: {np.mean(file_sizes)/1024/1024:.1f} MB")
print(f"  Total: {sum(file_sizes)/1024/1024/1024:.2f} GB")

# 다운스케일 버전 확인
for scale in ["images_2", "images_4", "images_8"]:
    scale_dir = os.path.join(SCENE_DIR, scale)
    if os.path.isdir(scale_dir):
        sample = glob.glob(os.path.join(scale_dir, "*"))
        if sample:
            sw, sh = Image.open(sample[0]).size
            print(f"\n{scale}/: {len(sample)}장, {sw}x{sh}")

In [ ]:
# === 셀 3: 카메라 파라미터 (cameras.bin) ===

CAMERA_MODEL_NAMES = {
    0: ("SIMPLE_PINHOLE", ["f", "cx", "cy"]),
    1: ("PINHOLE", ["fx", "fy", "cx", "cy"]),
    2: ("SIMPLE_RADIAL", ["f", "cx", "cy", "k"]),
    3: ("RADIAL", ["f", "cx", "cy", "k1", "k2"]),
    4: ("OPENCV", ["fx", "fy", "cx", "cy", "k1", "k2", "p1", "p2"]),
    5: ("OPENCV_FISHEYE", ["fx", "fy", "cx", "cy", "k1", "k2", "k3", "k4"]),
}

def read_next_bytes(fid, num_bytes, fmt, endian="<"):
    return struct.unpack(endian + fmt, fid.read(num_bytes))

sparse_dir = os.path.join(SCENE_DIR, "sparse", "0")
cameras_path = os.path.join(sparse_dir, "cameras.bin")

print(f"=== cameras.bin ===\n")

with open(cameras_path, "rb") as f:
    num_cameras = read_next_bytes(f, 8, "Q")[0]
    print(f"카메라 수: {num_cameras}\n")

    for _ in range(num_cameras):
        cam_id, model_id = read_next_bytes(f, 8, "ii")
        width, height = read_next_bytes(f, 16, "QQ")
        model_name, param_names = CAMERA_MODEL_NAMES.get(model_id, (f"UNKNOWN({model_id})", []))
        num_params = len(param_names)
        params = read_next_bytes(f, 8 * num_params, "d" * num_params)

        print(f"Camera ID: {cam_id}")
        print(f"  Model     : {model_name}")
        print(f"  Resolution: {width} x {height}")
        print(f"  Parameters:")
        for name, val in zip(param_names, params):
            print(f"    {name:4s} = {val:.6f}")
        print()

In [ ]:
# === 셀 4: 등록된 이미지 수 & 3D 포인트 수 (images.bin, points3D.bin) ===

images_path = os.path.join(sparse_dir, "images.bin")
points_path = os.path.join(sparse_dir, "points3D.bin")

with open(images_path, "rb") as f:
    num_images = read_next_bytes(f, 8, "Q")[0]

with open(points_path, "rb") as f:
    num_points = read_next_bytes(f, 8, "Q")[0]

total_images = len(img_files)

print(f"=== Reconstruction 요약 ===\n")
print(f"원본 이미지   : {total_images}장")
print(f"등록된 이미지  : {num_images}장 ({100*num_images/total_images:.1f}%)")
print(f"3D 포인트     : {num_points:,}개")

In [ ]:
# === 셀 5: 샘플 이미지 미리보기 ===
import matplotlib.pyplot as plt

n_show = min(6, len(img_files))
fig, axes = plt.subplots(1, n_show, figsize=(4 * n_show, 4))

# 균등 간격으로 샘플링
indices = np.linspace(0, len(img_files) - 1, n_show, dtype=int)
for i, idx in enumerate(indices):
    img = Image.open(img_files[idx])
    axes[i].imshow(img)
    axes[i].set_title(f"{os.path.basename(img_files[idx])}\n{img.size[0]}x{img.size[1]}", fontsize=9)
    axes[i].axis("off")

plt.suptitle(f"{SCENE_NAME} - 샘플 이미지", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# === 셀 6: 전체 씬 비교 ===

print(f"{'Scene':<12} {'Images':>8} {'Resolution':>14} {'Camera Model':>14} {'Registered':>12} {'3D Points':>12}")
print("-" * 80)

for scene in sorted(os.listdir(DATASET_BASE)):
    scene_path = os.path.join(DATASET_BASE, scene)
    if not os.path.isdir(scene_path):
        continue

    # 이미지 수 & 해상도
    img_path = os.path.join(scene_path, "images")
    if not os.path.isdir(img_path):
        continue
    imgs = glob.glob(os.path.join(img_path, "*"))
    n_imgs = len(imgs)
    if imgs:
        sample = Image.open(imgs[0])
        res = f"{sample.size[0]}x{sample.size[1]}"
    else:
        res = "-"

    # cameras.bin
    cam_file = os.path.join(scene_path, "sparse", "0", "cameras.bin")
    model_name = "-"
    if os.path.exists(cam_file):
        with open(cam_file, "rb") as f:
            f.read(8)
            _, model_id = struct.unpack("<ii", f.read(8))
            model_name = CAMERA_MODEL_NAMES.get(model_id, (f"?({model_id})",))[0]

    # images.bin, points3D.bin
    img_bin = os.path.join(scene_path, "sparse", "0", "images.bin")
    pts_bin = os.path.join(scene_path, "sparse", "0", "points3D.bin")
    n_reg = "-"
    n_pts = "-"
    if os.path.exists(img_bin):
        with open(img_bin, "rb") as f:
            n_reg = struct.unpack("<Q", f.read(8))[0]
    if os.path.exists(pts_bin):
        with open(pts_bin, "rb") as f:
            n_pts = struct.unpack("<Q", f.read(8))[0]

    reg_str = f"{n_reg}" if isinstance(n_reg, int) else n_reg
    pts_str = f"{n_pts:,}" if isinstance(n_pts, int) else n_pts

    print(f"{scene:<12} {n_imgs:>8} {res:>14} {model_name:>14} {reg_str:>12} {pts_str:>12}")